11/24/2026 Gottlieb

### 1. Updates

It has been some time since the last notebook iteration and it is important to report what progress the group has made. Project conventions are located in [DEFINITIONS](../../../docs/DEFINITIONS.md), progress and objectives are being tracked in [STATUS](../../../docs/STATUS.md), and the outline for the simulation workflow is in [simulation_phase_overview](../../../docs/simulation_phase_overview.md). The purpose of this notebook is to track what has changed since the previous plan, set objectives for the following days, understand the code structure to be used for simulations, plan the coming mini-project step-by-step, and any further directions.

We had previously outlined a rough plan to test the algorithm's power at detecting planted drift in comparison with cointegration. This experiment had not been run as the outline was not clear. The updated repository now gives this plan firmer grounding. Specifically we are answering claim C11 in the STATUS file.

### 2. Objectives

**Overall Project**: V&F propose a method for detecting an arbitrage signal. This first finds a low-volatility combination of assets, then multiples that portfolio's return in one time step by its return in the next. We want to find out what makes this detector value positive or negative. The first phase of this project is simulation. Simulated data lets us analyze one cause at a time and check what the detector responds. The second phase of the project (to be completed in the Spring) will be to test this metric on real market data.

**Mini-Project**: Directly test how well does the detector respond to an average return. This will be simulated by a combination two risky assets and cash. The risky assets share a random market movement, while their difference (their "spread") has a small average return chosen in advance (this is the "planted drift"). Because this return is planted, we can determine how large must this drift be before the V&F detector reliably notices it.

This act of noticing will be done with statistical precision. First, we will run many simulations with zero planted drift and set a cutoff that flags about 5% of these runs (a false-positive). Keeping the cutoff fixed, the simulation will be repeated with several nonzero drifts and we will count the fraction flagged at each drift size. This fraction is the "statistical power." For instance, 80% power means the method flags about 8 out of 10 independent runs at a specific drift size.

The average return of the spread will also be tested in a similar fashion. This is to show whether any loss of power comes from the detector statistic itself, and to check how the answer changes with noise and sample length.

The goal is to plot detection rate against planted-drift size. We will also check that the detector's exepcted average reading equals the square of the planted drift. This is just a brief overview of the task ahead, it will be explained in more detail below.

### 3. Code Map

The [simulation_phase_overview](../../../docs/simulation_phase_overview.md) organizes each experiment into four pieces to maintain consistency across the project: 1) a market generator with an exact truth, 2) prediction calculation, 3) the detector implementation, 4) a runner that repeats paths and saves results. Let's break this down one at a time.

**1) Market generator plus truth**
- Add a new module for this drift model (`results/planted_drift/market_planted_drift.py`) and implement the following methods.
- `simulate(p, T, seed)` returns a (T, 2) array of per-step log-returns (T total steps) for the two risky assets. Do this by generating a common shock, independent asset noise, and the chosen spread drift. 
- `truth(p)` returns (G0, G1) which G0 is the same step covariance and G1 is the lag-1 covariance matrix.

**2) Prediction calculator**
- `predict(G0, G1)` finds the true calm portfolio given the results from `truth()` 
- This can be called with `vf.predic.predict(G0, G1)` which was specially created to find the population's calm direction for two risky assets plus cash. 

**3) Detector wrapper**
- `run_detector(R, L=250, refresh=5, n_dirs=1, use_cash=True)` takes in the returns of the two risky assets and passes into the detector. For each path return the `products` (per-step products of consecutive calm-portfolio returns), `port_ret` (the calm portfolio's return at each step), and `weights` (the weights picked at each refresh)
- When complete, average the products to obtain one detector statistic per path. Alternatively, set `refresh=len(R)` to estiamte a direction only once.
- See `vf.detector.run_detector()` for the main call.

**4) Runner**
- Implement a runner method that loops over independent path seeds and drift settings then save one row per path with its settings and results.